# DatasetMain Browser

Use the controls below to pick any dataset under `/playpen-ssd/smerrill/deception2/DatasetMain` and inspect what was built. The notebook shows a dataset summary, manifest preview, one example record, any chat messages attached to that example, and the first few sentence rows for the same `example_id`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import textwrap

import pandas as pd
from IPython.display import display, Markdown

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
    WIDGET_IMPORT_ERROR = None
except Exception as exc:
    HAS_WIDGETS = False
    WIDGET_IMPORT_ERROR = exc

DATASET_ROOT = Path('/playpen-ssd/smerrill/deception2/DatasetMain')
DEFAULT_SENTENCE_ROWS = 12
MESSAGE_PREVIEW_CHARS = 240


In [ ]:
def count_jsonl_records(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def load_manifest(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl_slice(path: Path, start: int = 0, limit: int = 1) -> list[dict]:
    rows: list[dict] = []
    if not path.exists() or limit <= 0:
        return rows
    seen = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if seen < start:
                seen += 1
                continue
            rows.append(json.loads(line))
            seen += 1
            if len(rows) >= limit:
                break
    return rows


def find_sentences_for_example(path: Path, example_id: str, limit: int = DEFAULT_SENTENCE_ROWS) -> list[dict]:
    rows: list[dict] = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            if row.get("example_id") != example_id:
                continue
            rows.append(row)
            if len(rows) >= limit:
                break
    return rows


def summarize_messages(messages: list[dict]) -> pd.DataFrame:
    rows = []
    for idx, msg in enumerate(messages):
        content = msg.get("content")
        if isinstance(content, list):
            parts = []
            for item in content:
                if isinstance(item, dict):
                    text = item.get("text") or item.get("content") or str(item)
                else:
                    text = str(item)
                parts.append(text)
            content_text = "\n".join(parts)
        else:
            content_text = "" if content is None else str(content)
        preview = textwrap.shorten(" ".join(content_text.split()), width=MESSAGE_PREVIEW_CHARS, placeholder="...")
        rows.append({
            "message_idx": idx,
            "role": msg.get("role", ""),
            "preview": preview,
        })
    return pd.DataFrame(rows)


def build_dataset_index(root: Path = DATASET_ROOT) -> tuple[pd.DataFrame, dict[str, dict]]:
    rows = []
    index: dict[str, dict] = {}
    for env_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        for model_dir in sorted(p for p in env_dir.iterdir() if p.is_dir()):
            examples_path = model_dir / "examples.jsonl"
            sentences_path = model_dir / "sentences.jsonl"
            manifest_path = model_dir / "manifest.json"
            localization_dir = model_dir / "localization"
            dataset_id = f"{env_dir.name}/{model_dir.name}"
            manifest = load_manifest(manifest_path)
            first_example = read_jsonl_slice(examples_path, start=0, limit=1)
            first_example = first_example[0] if first_example else {}
            info = {
                "dataset_id": dataset_id,
                "environment": env_dir.name,
                "model": model_dir.name,
                "dataset_dir": model_dir,
                "examples_path": examples_path,
                "sentences_path": sentences_path,
                "manifest_path": manifest_path,
                "localization_dir": localization_dir,
                "examples_count": count_jsonl_records(examples_path),
                "sentences_count": count_jsonl_records(sentences_path),
                "has_manifest": manifest_path.exists(),
                "localization_json_count": sum(1 for p in localization_dir.glob("*.json")) if localization_dir.exists() else 0,
                "has_messages": "messages" in first_example,
                "strict_label_reason": first_example.get("strict_label_reason"),
                "text_field": manifest.get("text_field"),
                "input_root": manifest.get("input_root"),
            }
            rows.append({k: v for k, v in info.items() if k not in {"dataset_dir", "examples_path", "sentences_path", "manifest_path", "localization_dir"}})
            index[dataset_id] = info
    df = pd.DataFrame(rows).sort_values(["environment", "model"]).reset_index(drop=True)
    return df, index


DATASET_DF, DATASET_INDEX = build_dataset_index()
if DATASET_DF.empty:
    raise RuntimeError(f"No datasets found under {DATASET_ROOT}")

display(Markdown("## Dataset Summary"))
display(DATASET_DF)


In [ ]:
def render_dataset(dataset_id: str, example_index: int = 0, sentence_limit: int = DEFAULT_SENTENCE_ROWS, show_messages: bool = True) -> None:
    info = DATASET_INDEX[dataset_id]
    example_index = max(0, min(int(example_index), max(info["examples_count"] - 1, 0)))
    sentence_limit = max(1, int(sentence_limit))

    display(Markdown(f"## Preview: `{dataset_id}`"))

    summary_df = pd.DataFrame([{
        "environment": info["environment"],
        "model": info["model"],
        "examples_count": info["examples_count"],
        "sentences_count": info["sentences_count"],
        "has_manifest": info["has_manifest"],
        "localization_json_count": info["localization_json_count"],
        "has_messages": info["has_messages"],
        "strict_label_reason": info["strict_label_reason"],
        "text_field": info["text_field"],
    }])
    display(summary_df)

    manifest = load_manifest(info["manifest_path"])
    if manifest:
        display(Markdown("### Manifest"))
        display(pd.json_normalize(manifest, sep="."))

    examples = read_jsonl_slice(info["examples_path"], start=example_index, limit=1)
    if not examples:
        display(Markdown("No examples found."))
        return

    example = examples[0]
    display(Markdown(f"### Example Record `{example_index}`"))
    example_preview = dict(example)
    messages = example_preview.pop("messages", None)
    truth_context = example_preview.get("truth_context")
    if isinstance(truth_context, dict):
        example_preview["truth_context"] = json.dumps(truth_context, indent=2, ensure_ascii=False)
    action = example_preview.get("action")
    if isinstance(action, dict):
        example_preview["action"] = json.dumps(action, indent=2, ensure_ascii=False)
    display(pd.DataFrame([{k: example_preview.get(k) for k in example_preview.keys()}]).T.rename(columns={0: "value"}))

    if show_messages and isinstance(messages, list) and messages:
        display(Markdown("### Messages Preview"))
        display(summarize_messages(messages))

    example_id = example.get("example_id") or example.get("record_id")
    sentence_rows = find_sentences_for_example(info["sentences_path"], example_id, limit=sentence_limit)
    display(Markdown(f"### Sentence Rows for `{example_id}`"))
    if sentence_rows:
        sentence_df = pd.DataFrame(sentence_rows)
        preferred_cols = [
            "sentence_idx",
            "sentence_text",
            "deceptive",
            "recorded_deceptive",
            "label_verified",
            "strict_label_reason",
            "start",
            "end",
        ]
        display(sentence_df[[c for c in preferred_cols if c in sentence_df.columns]])
    else:
        display(Markdown("No sentence rows found for this example."))


if HAS_WIDGETS:
    options = [
        (f"{row.environment} / {row.model} ({row.examples_count:,} examples)", row.dataset_id)
        for row in DATASET_DF.itertuples(index=False)
    ]
    dataset_dropdown = widgets.Dropdown(options=options, description="Dataset:", layout=widgets.Layout(width="70%"))
    example_slider = widgets.IntSlider(value=0, min=0, max=max(int(DATASET_DF.iloc[0]["examples_count"]) - 1, 0), step=1, description="Example #", continuous_update=False)
    sentence_slider = widgets.IntSlider(value=DEFAULT_SENTENCE_ROWS, min=1, max=40, step=1, description="Sentences", continuous_update=False)
    show_messages_box = widgets.Checkbox(value=True, description="Show messages")

    def _sync_example_slider(*_):
        info = DATASET_INDEX[dataset_dropdown.value]
        example_slider.max = max(info["examples_count"] - 1, 0)
        if example_slider.value > example_slider.max:
            example_slider.value = 0

    dataset_dropdown.observe(_sync_example_slider, names="value")
    _sync_example_slider()

    controls = widgets.VBox([dataset_dropdown, example_slider, sentence_slider, show_messages_box])
    out = widgets.interactive_output(
        render_dataset,
        {
            "dataset_id": dataset_dropdown,
            "example_index": example_slider,
            "sentence_limit": sentence_slider,
            "show_messages": show_messages_box,
        },
    )
    display(controls, out)
else:
    display(Markdown(f"`ipywidgets` is not available here: `{WIDGET_IMPORT_ERROR}`"))
    display(Markdown("Call `render_dataset(dataset_id, example_index=0)` manually using one of the dataset ids below."))
    display(DATASET_DF[["dataset_id", "examples_count", "sentences_count"]])
    render_dataset(DATASET_DF.iloc[0]["dataset_id"])
